# alternatives-retrieval

Fetch and tidy. This notebook may download, reshape, filter, join and check.
It may not compute a measure the analysis reports — every percent change,
ratio, regression and projection lives in `alternatives-analysis.ipynb`.

It writes to `data/analysis/`, which is this folder's working set: the
archive in `data/processed/` is not touched.

Four things come in:

| What | From | Network |
|---|---|---|
| The Colorado school archive, 1977–2024 | `data/processed/` in this folder | no |
| County population by single year of age, 1990–2060 | `2026-09-bvsd/data/raw/sdo/` (SDO Vintage 2024) | no |
| Block and tract geography, and the BVSD boundary | Census TIGER 2023 | yes |
| Population under 18 by block, 2020 | Census PL 94-171 | yes |

Every download is cached in `data/raw/geo/` with its size and checksum, and
skipped if it is already there.

In [1]:
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
ARCHIVE = HERE / "data" / "processed"
LOOKUPS = HERE / "data" / "lookups"
RAW_GEO = HERE / "data" / "raw" / "geo"
ANALYSIS = HERE / "data" / "analysis"
SDO = HERE.parent / "2026-09-bvsd" / "data" / "raw" / "sdo"
ANALYSIS.mkdir(parents=True, exist_ok=True)
RAW_GEO.mkdir(parents=True, exist_ok=True)

BVSD_CODE = "0480"          # Boulder Valley RE 2
BOULDER_FIPS = "013"        # Boulder County
pd.set_option("display.width", 130)
print(f"archive: {ARCHIVE}")
print(f"sdo:     {SDO}")

archive: /home/user/charting-boulder/2026-10-alternatives/data/processed
sdo:     /home/user/charting-boulder/2026-09-bvsd/data/raw/sdo


## 1. The archive

Five tables, as built. Nothing is recomputed here — the columns are read and
the row counts printed so a reader can see what the rest of the notebook is
standing on.

In [2]:
codes = {"district_code": str, "school_code": str, "ncessch": str, "leaid": str}
school_year = pd.read_csv(ARCHIVE / "school-year.csv", dtype=codes)
school_fte = pd.read_csv(ARCHIVE / "school-teacher-fte.csv", dtype=codes)
district_fte = pd.read_csv(ARCHIVE / "district-teacher-fte-cde.csv", dtype=codes)
district_year = pd.read_csv(ARCHIVE / "district-year.csv", dtype=codes)
schools = pd.read_csv(ARCHIVE / "schools.csv", dtype=codes)
grades = pd.read_csv(ARCHIVE / "school-enrollment-by-grade.csv", dtype=codes)

for name, frame in [("school-year", school_year), ("school-teacher-fte", school_fte),
                    ("district-teacher-fte-cde", district_fte),
                    ("district-year", district_year), ("schools", schools),
                    ("school-enrollment-by-grade", grades)]:
    span = f"{int(frame['year'].min())}-{int(frame['year'].max())}" if "year" in frame else "-"
    print(f"  {name:28s} {len(frame):>8,} rows  {span}")

  school-year                    65,841 rows  1986-2024
  school-teacher-fte             38,777 rows  2000-2024
  district-teacher-fte-cde        6,695 rows  1986-2024
  district-year                   7,852 rows  1977-2024
  schools                         2,723 rows  -
  school-enrollment-by-grade    647,881 rows  1986-2024


## 2. County population aged 5 to 17

The State Demography Office publishes county population by single year of
age from 1990 to 2060, marking each row an estimate or a forecast. 5 to 17
is the school-age band: it is what the district's own planning uses and what
the regression in `2025-09-enrollments` used, so keeping it makes the two
comparable.

The file is read from `2026-09-bvsd/`, where it is already committed with its
provenance, rather than copied: seventeen megabytes of the same numbers in
two folders is two things to keep in step.

In [3]:
sya = pd.read_csv(SDO / "sya-county.csv", skiprows=1, dtype={"countyfips": str})
sya["countyfips"] = sya["countyfips"].str.zfill(3)
school_age = (sya[sya["age"].between(5, 17)]
              .groupby(["countyfips", "county", "year", "datatype"], as_index=False)
              ["totalpopulation"].sum()
              .rename(columns={"totalpopulation": "pop_5_17"}))
# A county-year is an estimate or a forecast, never both.
assert school_age.groupby(["countyfips", "year"]).size().max() == 1

under5 = (sya[sya["age"].between(0, 4)]
          .groupby(["countyfips", "year"], as_index=False)["totalpopulation"].sum()
          .rename(columns={"totalpopulation": "pop_0_4"}))
school_age = school_age.merge(under5, on=["countyfips", "year"], how="left")

print(f"{len(school_age):,} county-years, {school_age['year'].min()}-{school_age['year'].max()}")
print(school_age.groupby("datatype")["year"].agg(["min", "max", "count"]))
boulder = school_age[school_age["countyfips"] == BOULDER_FIPS]
print("\nBoulder County, 5-17:")
print(boulder[boulder["year"].isin([1990, 2000, 2010, 2020, 2024, 2030, 2040, 2050, 2060])]
      [["year", "pop_5_17", "pop_0_4", "datatype"]].to_string(index=False))

4,615 county-years, 1990-2060
           min   max  count
datatype                   
Estimate  1990  2024   2275
Forecast  2025  2060   2340

Boulder County, 5-17:
 year  pop_5_17  pop_0_4 datatype
 1990     36200    15999 Estimate
 2000     45400    16624 Estimate
 2010     46415    16557 Estimate
 2020     48100    13184 Estimate
 2024     43997    12526 Estimate
 2030     38685    12526 Forecast
 2040     38270    14580 Forecast
 2050     41372    13676 Forecast
 2060     38207    12230 Forecast


## 3. The district panel

One row per district per year: how many pupils it taught, how many teachers
it employed, how many schools it ran, and how many 5-to-17-year-olds lived
in its county. This is the table the staffing model is fitted on.

Three joins, and each one can fail quietly, so each is counted:

- **Staffing** comes from `district-teacher-fte-cde.csv`, which carries what
  CDE states where it states it and the sum of the district's schools
  everywhere else. The two are kept apart here as well, because they are not
  the same measurement.
- **Schools** are counted from the school panel, which covers every year
  through NCES. The yearbooks' own published count is carried alongside for
  1986–1999 as a check rather than a substitute.
- **County** comes from `data/lookups/district-county.csv`, built by the
  pipeline from every source that states one.

In [4]:
crosswalk = pd.read_csv(LOOKUPS / "district-county.csv", dtype={"district_code": str})

# The school panel carries a CDE district code only from 2004: before that it
# is NCES-only, and NCES does not use CDE codes. The district *name* is there
# in every one of the thirty-nine years, so the archive's own name crosswalk
# fills the gap - and it matters, because without it the panel cannot see that
# Boulder Valley ran 60 schools in 2000 and 53 in 2004.
# The crosswalk is keyed on the archive's own normalised district name, which
# strips the organisational suffix ("Academy 20" -> ACADEMY). Reimplementing
# that here would be a second copy of a rule that has already caught one bug
# (Garfield RE-2 and Garfield 16 are different districts), so the pipeline's
# own functions are imported instead.
from pipeline.normalize import district_key, index_district_names

name_to_code = json.loads((LOOKUPS / "district-crosswalk.json").read_text())
index_district_names(
    [{"district_name": entry["district_name"], "county_name": ""}
     for entry in name_to_code.values()])
lookup = {key: entry["district_code"] for key, entry in name_to_code.items()}

def district_of(name: str) -> str | float:
    if not isinstance(name, str):
        return np.nan
    return lookup.get(district_key(name), np.nan)

school_year["district_by_name"] = school_year["district_name"].map(district_of)
school_year["district"] = school_year["district_code"].fillna(school_year["district_by_name"])
filled = school_year["district_code"].isna() & school_year["district"].notna()
print(f"school-years with a district code in the file:  "
      f"{school_year['district_code'].notna().sum():,}")
print(f"  filled from the district name:                {filled.sum():,}")
print(f"  still unattributed:                           {school_year['district'].isna().sum():,}")

school_counts = (school_year[school_year["district"].notna()]
                 .groupby(["year", "district"], as_index=False)
                 .agg(schools_open=("school_code", "nunique"),
                      enrollment_panel=("enrollment_total_cde", "sum"),
                      enrollment_ccd=("enrollment_total_ccd", "sum"))
                 .rename(columns={"district": "district_code"}))

panel = district_fte.merge(school_counts, on=["year", "district_code"], how="outer")
panel = panel.merge(crosswalk[["district_code", "county_name"]]
                    .rename(columns={"county_name": "county_crosswalk"}),
                    on="district_code", how="left")
# The yearbook era writes counties in capitals and the SDO file in title
# case, so the join has to be told they are the same word.
panel["county"] = (panel["county_crosswalk"].fillna(panel["county_name"])
                   .str.strip().str.title())

# The staffing measure: what CDE states, falling back to the sum of the
# district's own schools. Which one a row used is kept, because a mixed series
# that does not say so is a trap.
panel["teacher_fte"] = panel["teacher_fte_published"].fillna(panel["teacher_fte_school_sum"])
panel["teacher_fte_basis"] = np.where(panel["teacher_fte_published"].notna(), "published",
                             np.where(panel["teacher_fte_school_sum"].notna(), "school_sum", ""))
# Three ways to count a district's schools, in order of how directly they
# were observed. The school panel is the spine, but it counts a school to a
# district only where the source names one - and in 2001 to 2003 the panel is
# NCES-only, which carries no CDE district code. Those years would have
# dropped out of the model entirely, and nothing about the row would have
# shown why.
panel["schools"] = (panel["schools_open"]
                    .fillna(panel["schools_in_sum"])
                    .fillna(panel["schools_published"]))
panel["schools_basis"] = np.where(panel["schools_open"].notna(), "school_panel",
                         np.where(panel["schools_in_sum"].notna(), "fte_file",
                         np.where(panel["schools_published"].notna(), "published", "")))
panel["enrollment"] = panel["enrollment_published"].fillna(panel["enrollment_panel"])

panel = panel.merge(school_age[["countyfips", "county", "year", "pop_5_17", "pop_0_4", "datatype"]],
                    left_on=["county", "year"], right_on=["county", "year"], how="left")

matched = panel["pop_5_17"].notna().mean()
print(f"district-years: {len(panel):,}")
print(f"  with a county population joined: {matched:.1%}")
print(f"  with teacher FTE:  {panel['teacher_fte'].notna().mean():.1%}"
      f"   with a school count: {panel['schools'].notna().mean():.1%}")
missing = panel[panel["pop_5_17"].isna()]
print("\nunjoined, by reason:")
print(f"  before 1990, where the SDO series starts: "
      f"{(missing['year'] < 1990).sum():,}")
print(f"  no county named at all:                   "
      f"{missing['county'].isna().sum():,}")
other = missing[(missing["year"] >= 1990) & missing["county"].notna()]
print(f"  a county the SDO file does not carry:     {len(other):,}")
if len(other):
    print("   ", sorted(other["county"].unique())[:8])

school-years with a district code in the file:  38,492
  filled from the district name:                24,063
  still unattributed:                           3,286
district-years: 7,326
  with a county population joined: 84.4%
  with teacher FTE:  88.7%   with a school count: 99.9%

unjoined, by reason:
  before 1990, where the SDO series starts: 777
  no county named at all:                   220
  a county the SDO file does not carry:     150
    ['Colorado Boces', 'Colorado Bocs', 'None']


### The panel, as it will be modelled

A district-year enters the model only if it has staffing, a school count and
a county population. BOCES are dropped: they are shared agencies, not
districts, and they do not run a county's schools.

In [5]:
# The panel keeps every district-year that can be placed in a county, whether
# or not it has staffing. Filtering to complete rows here would hide which
# years the staffing model is silent about, and the missing years are not
# random: CDE published no ratio report for 1999, 2017, 2020 or 2021.
model_panel = panel[(panel["unit_type"] != "boces")
                    & panel["pop_5_17"].notna()
                    & panel["district_code"].notna()
                    & (panel["district_code"] != "")].copy()
model_panel = model_panel.sort_values(["district_code", "year"])
keep = ["year", "district_code", "district_name", "county", "countyfips", "unit_type",
        "teacher_fte", "teacher_fte_basis", "teacher_fte_published", "teacher_fte_school_sum",
        "schools", "schools_basis", "schools_published", "schools_open",
        "enrollment", "enrollment_panel", "enrollment_ccd",
        "pop_5_17", "pop_0_4", "datatype"]
model_panel = model_panel[keep]
model_panel.to_csv(ANALYSIS / "district-panel.csv", index=False)
print(f"wrote data/analysis/district-panel.csv: {len(model_panel):,} rows, "
      f"{model_panel['year'].min()}-{model_panel['year'].max()}, "
      f"{model_panel['district_code'].nunique()} districts")
coverage = (model_panel.assign(has_fte=model_panel["teacher_fte"].notna(),
                               has_schools=model_panel["schools"].notna())
            .groupby("year")[["has_fte", "has_schools"]].sum().astype(int))
coverage["districts"] = model_panel.groupby("year").size()
print("\ndistricts per year, and how many carry each measure:")
print(coverage.to_string())

wrote data/analysis/district-panel.csv: 5,989 rows, 1990-2024, 180 districts

districts per year, and how many carry each measure:
      has_fte  has_schools  districts
year                                 
1990      160          164        164
1991      164          164        164
1992      165          165        165
1993      165          165        165
1994      165          165        165
1995      165          165        165
1996      166          166        166
1997      163          166        166
1998      157          166        166
1999        0          166        166
2000      166          166        166
2001      166          166        166
2002      166          166        166
2003      166          166        166
2004      178          178        178
2005      178          178        178
2006      178          178        178
2007      178          178        178
2008      177          178        178
2009      178          179        179
2010      166          166       

## 4. Closures

The archive labels a school's disappearance from the panel from the data
alone, and its confidence column is the honest part of that table: a
closure, a merger, a rename and a code change can look alike. Those labels
carry the statewide picture. Boulder's own closures are checked by hand
further down, because that is where the argument rests on individual
buildings.

In [6]:
closures = schools.copy()
closures["last_year"] = closures["last_year"].astype(int)
closures["first_year"] = closures["first_year"].astype(int)
print(closures.groupby(["closure_label", "closure_confidence"]).size()
      .sort_values(ascending=False).to_string())

# A school's district is the last one it appears in.
last_seen = (school_year.sort_values("year").groupby("ncessch")
             .agg(last_district=("district_code", "last"),
                  last_name=("school_name", "last"),
                  last_enrollment=("enrollment_total_cde", "last"),
                  latitude=("latitude", "last"), longitude=("longitude", "last")))
closures = closures.merge(last_seen, on="ncessch", how="left")
closures.to_csv(ANALYSIS / "school-registry.csv", index=False)
print(f"\nwrote data/analysis/school-registry.csv: {len(closures):,} schools")

closure_label  closure_confidence
still_open     high                  1934
closed         high                   614
renamed        low                    101
closed         medium                  57
code_changed   low                     17

wrote data/analysis/school-registry.csv: 2,723 schools


### Size in the years before a school disappears

The event table: for every school in the panel, its enrollment and staffing
in each of the five years before its last, and whether it disappeared. This
is what the closure model is fitted on, and what the event study reads.

In [7]:
sy = school_year[["year", "ncessch", "school_code", "district", "school_name",
                  "enrollment_total_cde", "enrollment_total_ccd",
                  "teacher_fte_cde", "teacher_fte_ccd", "status"]].copy()
sy["enrollment"] = sy["enrollment_total_cde"].fillna(sy["enrollment_total_ccd"])
sy = sy.rename(columns={"district": "district_code"})
sy["teacher_fte"] = sy["teacher_fte_cde"].fillna(sy["teacher_fte_ccd"])
sy = sy.merge(closures[["ncessch", "closure_label", "closure_confidence", "last_year"]],
              on="ncessch", how="left")
# Whether a disappearance counts as a closure, and which year it should be
# measured in, are analysis choices and are made in the analysis notebook.
# What is carried here is the raw fact: the years a school is observed, and
# what the registry labels its disappearance.
sy = sy.sort_values(["ncessch", "year"])
sy["years_to_last"] = sy["last_year"] - sy["year"]
sy.to_csv(ANALYSIS / "school-panel.csv", index=False)
print(f"wrote data/analysis/school-panel.csv: {len(sy):,} school-years")
print(f"  schools: {sy['ncessch'].nunique():,};  labelled closed or merged: "
      f"{closures['closure_label'].isin(['closed', 'merged']).sum():,}")
print(f"  panel ends {int(sy['year'].max())}")

wrote data/analysis/school-panel.csv: 66,828 school-years
  schools: 2,701;  labelled closed or merged: 671
  panel ends 2024


## 5. Geography

Where the children are, and where the schools are.

Everything here is keyless and public. The City of Boulder's neighbourhood
boundaries were the intended layer for local texture; its open-data portal
returned HTTP 522 throughout and its ArcGIS server publishes no neighbourhood
service, so **census block groups and the district's municipalities stand in
for neighbourhoods**. That is a real substitution and it is recorded here
rather than in a footnote: block groups are drawn for population balance, not
for how Boulder thinks about itself.

In [8]:
import urllib.request

USER_AGENT = "charting-boulder/2026-10-alternatives (+https://github.com/brianckeegan/charting-boulder)"
SOURCES = {
    "tl_2023_08_unsd.zip":
        "https://www2.census.gov/geo/tiger/TIGER2023/UNSD/tl_2023_08_unsd.zip",
    "tl_2023_08_tract.zip":
        "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_08_tract.zip",
    "tl_2023_08_bg.zip":
        "https://www2.census.gov/geo/tiger/TIGER2023/BG/tl_2023_08_bg.zip",
    "tl_2023_08_place.zip":
        "https://www2.census.gov/geo/tiger/TIGER2023/PLACE/tl_2023_08_place.zip",
    "tl_2020_08_tabblock20.zip":
        "https://www2.census.gov/geo/tiger/TIGER2020/TABBLOCK20/tl_2020_08_tabblock20.zip",
    "co2020.pl.zip":
        "https://www2.census.gov/programs-surveys/decennial/2020/data/"
        "01-Redistricting_File--PL_94-171/Colorado/co2020.pl.zip",
}


def fetch(name: str, url: str, tries: int = 4) -> Path:
    dest = RAW_GEO / name
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    delay = 2
    for attempt in range(tries):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(request, timeout=900) as response:
                dest.write_bytes(response.read())
            return dest
        except Exception as exc:  # noqa: BLE001
            if attempt == tries - 1:
                raise
            import time
            time.sleep(delay)
            delay *= 2
    raise RuntimeError("unreachable")


manifest_path = RAW_GEO / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
for name, url in SOURCES.items():
    path = fetch(name, url)
    if name not in manifest:
        manifest[name] = {
            "url": url, "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    print(f"  {name:32s} {path.stat().st_size / 1e6:8.1f} MB")
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

  tl_2023_08_unsd.zip                   2.1 MB
  tl_2023_08_tract.zip                  8.4 MB
  tl_2023_08_bg.zip                    13.5 MB
  tl_2023_08_place.zip                  2.8 MB
  tl_2020_08_tabblock20.zip           218.5 MB
  co2020.pl.zip                        24.2 MB


1375

### The district boundary, and what is inside it

BVSD is not Boulder. The district runs from Nederland and the mountain towns
through Boulder, Louisville, Lafayette and Superior, and those places are not
moving the same way — which is the whole reason a closure plan drawn on the
city alone would miss something.

In [9]:
import geopandas as gpd

unsd = gpd.read_file(f"zip://{RAW_GEO / 'tl_2023_08_unsd.zip'}")
bvsd = unsd[unsd["NAME"].str.contains("Boulder Valley", case=False, na=False)].to_crs(4326)
assert len(bvsd) == 1, f"expected one BVSD boundary, found {len(bvsd)}"
bvsd_geom = bvsd.geometry.iloc[0]
# EPSG:6428 is Colorado North in US survey feet, so the divisor is square
# feet per square mile, not square metres.
area_sq_mi = bvsd.to_crs(6428).geometry.area.iloc[0] / 27_878_400
print(f"{bvsd['NAME'].iloc[0]} (GEOID {bvsd['GEOID'].iloc[0]}): {area_sq_mi:,.0f} square miles")

places = gpd.read_file(f"zip://{RAW_GEO / 'tl_2023_08_place.zip'}").to_crs(4326)
in_district = places[places.geometry.intersects(bvsd_geom)].copy()
in_district["share_in_bvsd"] = (
    in_district.geometry.intersection(bvsd_geom).to_crs(6428).area
    / in_district.to_crs(6428).geometry.area)
in_district = in_district[in_district["share_in_bvsd"] > 0.02]
# A place that only touches the boundary is not in the district.
in_district = in_district[
    in_district.geometry.intersection(bvsd_geom).to_crs(6428).area > 0]
print(f"\n{len(in_district)} municipalities overlap the district:")
print(in_district.sort_values("share_in_bvsd", ascending=False)
      [["NAME", "share_in_bvsd"]].to_string(index=False, float_format=lambda v: f"{v:.0%}"))

bvsd[["GEOID", "NAME", "geometry"]].to_file(ANALYSIS / "bvsd-boundary.geojson", driver="GeoJSON")
in_district[["GEOID", "NAME", "share_in_bvsd", "geometry"]].to_file(
    ANALYSIS / "bvsd-places.geojson", driver="GeoJSON")

Boulder Valley School District RE-2 (GEOID 0802490): 482 square miles

31 municipalities overlap the district:
                    NAME  share_in_bvsd
               Lafayette           100%
                Glendale           100%
              Bark Ranch           100%
        Eldorado Springs           100%
                 Valmont           100%
                 Crisman           100%
            Rollinsville           100%
               Gold Hill           100%
             Hidden Lake           100%
              Lazy Acres           100%
                Sunshine           100%
                  Eldora           100%
              Louisville           100%
Bonanza Mountain Estates           100%
                    Ward           100%
        Mountain Meadows           100%
             Tall Timber           100%
               Jamestown           100%
         Paragon Estates           100%
                  Leyner           100%
               Sugarloaf           100%
         

### Children by block

The 2020 census redistricting file gives population and population 18 and
over for every block; the difference is the children. Block grain matters
here because a school catchment is smaller than a tract, and because the
blocks a closure would move children between are often adjacent.

This is a count of children, not of pupils: it includes those at private
schools, charters and home, and it is a 2020 snapshot. It is used to say
where children live relative to buildings, never how many a school enrolls —
that comes from the archive.

In [10]:
with zipfile.ZipFile(RAW_GEO / "co2020.pl.zip") as archive:
    names = archive.namelist()
    geo_name = next(n for n in names if n.lower().endswith("geo2020.pl"))
    seg1_name = next(n for n in names if n.lower().endswith("012020.pl"))
    # Segment 2 carries P3 and P4 - the tables for population 18 and over.
    # Segment 3 is H1, occupancy status, and reading it as if it were P3 makes
    # 98% of Colorado a child.
    seg2_name = next(n for n in names if n.lower().endswith("022020.pl"))
    geo = pd.read_csv(io.BytesIO(archive.read(geo_name)), sep="|", header=None,
                      dtype=str, encoding="latin-1", low_memory=False)
    seg1 = pd.read_csv(io.BytesIO(archive.read(seg1_name)), sep="|", header=None,
                       dtype=str, encoding="latin-1", low_memory=False)
    seg2 = pd.read_csv(io.BytesIO(archive.read(seg2_name)), sep="|", header=None,
                       dtype=str, encoding="latin-1", low_memory=False)

# PL 94-171 layout: the geoheader's fields are positional. LOGRECNO joins the
# header to the data segments; SUMLEV 750 is the block.
geo = geo.rename(columns={7: "LOGRECNO", 2: "SUMLEV", 9: "GEOCODE"})
blocks_geo = geo[geo["SUMLEV"] == "750"][["LOGRECNO", "GEOCODE"]].copy()
# Both segments put LOGRECNO fifth and the table's first cell sixth:
# P0010001 is the total population, P0030001 the population 18 and over.
seg1_total = seg1.rename(columns={4: "LOGRECNO", 5: "P0010001"})[["LOGRECNO", "P0010001"]]
seg2_total = seg2.rename(columns={4: "LOGRECNO", 5: "P0030001"})[["LOGRECNO", "P0030001"]]
blocks_pop = (blocks_geo.merge(seg1_total, on="LOGRECNO")
              .merge(seg2_total, on="LOGRECNO"))
blocks_pop["population"] = blocks_pop["P0010001"].astype(int)
blocks_pop["adults"] = blocks_pop["P0030001"].astype(int)
blocks_pop["children"] = blocks_pop["population"] - blocks_pop["adults"]
blocks_pop["GEOID20"] = blocks_pop["GEOCODE"].str[-15:]
print(f"Colorado blocks: {len(blocks_pop):,}; "
      f"population {blocks_pop['population'].sum():,}, "
      f"under 18 {blocks_pop['children'].sum():,}")

Colorado blocks: 140,345; population 5,773,714, under 18 1,264,138


In [11]:
blocks = gpd.read_file(f"zip://{RAW_GEO / 'tl_2020_08_tabblock20.zip'}",
                       columns=["GEOID20", "COUNTYFP20", "TRACTCE20", "ALAND20", "geometry"])
blocks = blocks[blocks["COUNTYFP20"].isin([BOULDER_FIPS, "014"])].to_crs(4326)
blocks = blocks.merge(blocks_pop[["GEOID20", "population", "adults", "children"]],
                      on="GEOID20", how="left")
inside = blocks[blocks.geometry.representative_point().within(bvsd_geom)].copy()
print(f"blocks in Boulder and Broomfield counties: {len(blocks):,}")
print(f"  of those, inside the BVSD boundary:      {len(inside):,}")
print(f"  children under 18 in the district (2020): {int(inside['children'].sum()):,}")
inside["block_group"] = inside["GEOID20"].str[:12]
inside["tract"] = inside["GEOID20"].str[:11]
inside[["GEOID20", "tract", "block_group", "population", "adults", "children",
        "ALAND20", "geometry"]].to_file(ANALYSIS / "bvsd-blocks.geojson", driver="GeoJSON")
print(f"wrote data/analysis/bvsd-blocks.geojson")

blocks in Boulder and Broomfield counties: 7,193
  of those, inside the BVSD boundary:      4,133
  children under 18 in the district (2020): 41,063


wrote data/analysis/bvsd-blocks.geojson


### The schools themselves

Location, grade span and the most recent enrollment by grade, for every BVSD
school in the archive's last year. Coordinates come from the NCES directory
and are carried backward, so a school that moved buildings carries its later
location — which is stated in the archive's limitations and matters here,
because these coordinates are used for distance.

In [12]:
last_year = int(school_year["year"].max())
bvsd_schools = school_year[(school_year["district_code"] == BVSD_CODE)
                           & (school_year["year"] == last_year)].copy()
bvsd_grades = grades[(grades["district_code"] == BVSD_CODE)
                     & (grades["year"] == last_year)]
wide = (bvsd_grades.pivot_table(index="school_code", columns="grade",
                                values="enrollment_cde", aggfunc="sum")
        .rename(columns=str))
order = ["PK", "K"] + [str(g) for g in range(1, 13)]
wide = wide.reindex(columns=[g for g in order if g in wide.columns])
bvsd_schools = bvsd_schools.merge(wide, on="school_code", how="left")

def span(row):
    present = [g for g in order if g in wide.columns and pd.notna(row.get(g)) and row.get(g) > 0]
    return f"{present[0]}-{present[-1]}" if present else ""

bvsd_schools["grade_span"] = bvsd_schools.apply(span, axis=1)
bvsd_schools["level"] = np.select(
    [bvsd_schools["grade_span"].str.startswith(("PK", "K")) & ~bvsd_schools[[g for g in ("9","10","11","12") if g in wide.columns]].gt(0).any(axis=1),
     bvsd_schools[[g for g in ("9","10","11","12") if g in wide.columns]].gt(0).any(axis=1)],
    ["elementary_or_middle", "has_high_grades"], default="other")
located = bvsd_schools["latitude"].notna().sum()
print(f"BVSD schools in {last_year}: {len(bvsd_schools)}   with coordinates: {located}")
print(f"total enrollment: {int(bvsd_schools['enrollment_total_cde'].sum()):,}")
bvsd_schools.to_csv(ANALYSIS / "bvsd-schools.csv", index=False)
print(bvsd_schools.sort_values("enrollment_total_cde", ascending=False)
      [["school_code", "school_name", "grade_span", "enrollment_total_cde", "teacher_fte_cde"]]
      .head(12).to_string(index=False))

BVSD schools in 2024: 56   with coordinates: 56
total enrollment: 27,991
school_code                 school_name grade_span  enrollment_total_cde  teacher_fte_cde
       0924         Boulder High School       9-12                1939.0             91.1
       2892        Fairview High School       9-12                1863.0             80.8
       1070      Broomfield High School       9-12                1694.0             77.9
       1380       Centaurus High School       9-12                1557.0             78.0
       6816 Peak to Peak Charter School       K-12                1447.0             81.8
       5999         Monarch High School       9-12                1431.0             64.5
       0441      Aspen Creek K-8 School       PK-8                 844.0             46.8
       2639           Meadowlark School       PK-8                 747.0             39.9
       6000          Monarch K-8 School       PK-8                 698.0             45.6
       5306    Louisville M

## 6. What is still missing

Written here rather than discovered halfway through the analysis.

In [13]:
gaps = pd.DataFrame([
    ("BVSD capacity by school", "the Resilient Schools presentation",
     "not supplied; utilisation cannot be computed against the district's own bar"),
    ("City of Boulder neighbourhood boundaries", "opendata.bouldercolorado.gov",
     "HTTP 522 throughout; block groups and municipalities stand in"),
    ("Teacher FTE, 1999 / 2017 / 2020 / 2021", "CDE",
     "not published; the staffing model is silent in those years"),
    ("County population before 1990", "SDO",
     "the series starts in 1990, so the panel does too"),
], columns=["what", "where_it_would_come_from", "status"])
print(gaps.to_string(index=False))
gaps.to_csv(ANALYSIS / "known-gaps.csv", index=False)

                                    what           where_it_would_come_from                                                                      status
                 BVSD capacity by school the Resilient Schools presentation not supplied; utilisation cannot be computed against the district's own bar
City of Boulder neighbourhood boundaries       opendata.bouldercolorado.gov               HTTP 522 throughout; block groups and municipalities stand in
  Teacher FTE, 1999 / 2017 / 2020 / 2021                                CDE                  not published; the staffing model is silent in those years
           County population before 1990                                SDO                            the series starts in 1990, so the panel does too


## 7. The proposal itself

Resolution 26-27 and the board work session of 15 September 2026, supplied
by hand and kept in `data/raw/proposal/` with their checksums. BoardDocs
refuses automated fetching (HTTP 403), so these cannot be re-downloaded by
this notebook; the manifest records what was read.

Two things are taken from them. The **actions** come from the resolution's
operative clauses, which name every school. The **figures** come from the
work session's slides, each one carried with the slide it is read off, so a
reader can check it.

In [14]:
PROPOSAL = HERE / "data" / "raw" / "proposal"
from pdfminer.high_level import extract_pages     # noqa: E402
from pdfminer.layout import LTTextContainer       # noqa: E402

def slide_text(path: Path) -> list[str]:
    out = []
    for layout in extract_pages(str(path)):
        out.append("\n".join(e.get_text().strip() for e in layout
                             if isinstance(e, LTTextContainer)))
    return out

resolution = slide_text(PROPOSAL / "resolution-26-27-resilient-schools-2026-09-15.pdf")
session = slide_text(PROPOSAL / "boe-work-session-resilient-schools-2026-09-15.pdf")
(ANALYSIS / "proposal-resolution.txt").write_text("\n\f\n".join(resolution))
(ANALYSIS / "proposal-work-session.txt").write_text("\n\f\n".join(session))
print(f"resolution: {len(resolution)} pages;  work session: {len(session)} slides")

resolution: 5 pages;  work session: 61 slides


### What the proposal does

Transcribed from the resolution's operative clauses. Every school name here
appears verbatim in the document; the archive's code for it is matched by
name and printed so a mismatch is visible rather than silent.

In [15]:
ACTIONS = [
    # region, school, action, detail
    ("Broomfield", "Birch Elementary School", "close",
     "merge into Kohl; ICAN program relocates to Kohl"),
    ("Broomfield", "Kohl Elementary School", "receive",
     "one attendance area with Birch"),
    ("Louisville and Superior", "Monarch K-8 School", "reconfigure",
     "elementary programme closes; becomes a 6-8 middle school"),
    ("Louisville and Superior", "Eldorado K-8 School", "reconfigure",
     "middle programme closes; becomes a PK-5 elementary school"),
    ("Louisville and Superior", "Fireside Elementary School", "receive",
     "takes part of Monarch K-5's attendance area"),
    ("Louisville and Superior", "Superior Elementary School", "receive",
     "takes part of Monarch K-5's attendance area"),
    ("Boulder", "Douglass Elementary School", "close",
     "closes as a neighbourhood school; High Peaks moves into the building"),
    ("Boulder", "High Peaks Elementary School", "relocate",
     "moves into the Douglass building"),
    ("Boulder", "Community Montessori School", "close",
     "building closes; programme co-locates with BCSIS in the Aurora 7 building"),
    ("Boulder", "Boulder Community School/Integrated Studies", "receive",
     "Community Montessori co-locates"),
    ("Boulder", "Coal Creek Elementary School", "receive", "takes part of Douglass's area"),
    ("Boulder", "Eisenhower Elementary School", "receive", "takes part of Douglass's area"),
    ("Boulder", "Heatherwood Elementary School", "receive",
     "takes part of Douglass's area; receives the ICAN programme"),
    ("Boulder", "Mesa Elementary School", "close",
     "merges into Bear Creek; RISE programme relocates to Bear Creek"),
    ("Boulder", "Bear Creek Elementary School", "receive", "one attendance area with Mesa"),
    ("Boulder", "Flatirons Elementary School", "close",
     "attendance area splits at Boulder Canyon Drive between Foothill and Whittier"),
    ("Boulder", "Foothill Elementary School", "receive", "takes north of Boulder Canyon Dr"),
    ("Boulder", "Whittier Elementary School", "receive",
     "takes south of Boulder Canyon Dr; receives RISE; bell schedule adjusted"),
]
actions = pd.DataFrame(ACTIONS, columns=["region", "school_name", "action", "detail"])
current = school_year[(school_year["district"] == BVSD_CODE)
                      & (school_year["year"] == last_year)].copy()
lookup = current.assign(key=current["school_name"].str.upper()).set_index("key")["school_code"]
actions["school_code"] = actions["school_name"].str.upper().map(lookup)
unmatched = actions[actions["school_code"].isna()]
print(f"{len(actions)} school actions; matched to the archive: "
      f"{actions['school_code'].notna().sum()}")
if len(unmatched):
    print("unmatched:", unmatched["school_name"].tolist())
actions.to_csv(ANALYSIS / "proposal-actions.csv", index=False)
print(actions.groupby(["region", "action"]).size().to_string())

18 school actions; matched to the archive: 18
region                   action     
Boulder                  close          4
                         receive        7
                         relocate       1
Broomfield               close          1
                         receive        1
Louisville and Superior  receive        2
                         reconfigure    2


### The figures the proposal turns on

Read off the work session slides, each with the slide it came from. These
are the district's own numbers and are never recomputed here — where the
analysis disagrees with one, both are shown.

In [16]:
FIGURES = [
    ("classes_per_grade_2025_26", 2.0, "average classes per grade, elementary", 5),
    ("classes_per_grade_2030_31", 2.5, "average classes per grade with the proposal", 5),
    ("schools_below_two_2025_26", 14, "elementary schools below two classes per grade", 5),
    ("schools_below_two_2030_31", 6, "the same, in 2030-31 with the proposal", 5),
    ("utilization_2025_26", 0.68, "district building utilisation", 5),
    ("utilization_2030_31", 0.75, "district building utilisation with the proposal", 5),
    ("students_per_class_round", 25, "a 'round' is one class of 25", 5),
    ("two_rounds_enrollment", 300, "two classes a grade is about 300 pupils", 5),
    ("schools_below_two_2030_no_action", 14, "if no action were taken, by 2030", None),
    ("decline_since_2017", 3675, "school-aged children lost since 2017", None),
    ("decline_to_2030", 1706, "further decline anticipated by 2030", None),
    ("neighbourhood_share_low", 0.67, "share attending their neighbourhood school", 21),
    ("neighbourhood_share_high", 0.70, "the same, upper end", 21),
    ("class_max_k1", 26, "negotiated class size maximum, grades K-1", 34),
    ("class_max_2_3", 29, "negotiated class size maximum, grades 2-3", 34),
    ("class_max_4_5", 31, "negotiated class size maximum, grades 4-5", 34),
    ("traveling_teachers_2024_25", 13, "specials teachers working across schools", 35),
    ("traveling_teachers_2025_26", 29, "specials teachers working across schools", 35),
    ("traveling_teachers_2026_27", 36, "specials teachers working across schools", 35),
    ("traveling_teachers_2027_28_no_action", 40, "projected without the proposal", 35),
    ("traveling_teachers_2027_28_with_plan", 18, "projected with the proposal", 35),
    ("split_positions_2027_28_no_action", 12, "posts with no home school, without the proposal", 35),
    ("split_positions_2027_28_with_plan", 0, "posts with no home school, with the proposal", 35),
]
figures = pd.DataFrame(FIGURES, columns=["name", "value", "what_it_is", "slide"])
figures["source"] = np.where(figures["slide"].notna(),
                             "BOE work session, 2026-09-15, slide " + figures["slide"].astype("Int64").astype(str),
                             "Resolution 26-27, 2026-09-15")
figures.to_csv(ANALYSIS / "proposal-figures.csv", index=False)
print(figures[["name", "value", "source"]].to_string(index=False))

                                name   value                                 source
           classes_per_grade_2025_26    2.00  BOE work session, 2026-09-15, slide 5
           classes_per_grade_2030_31    2.50  BOE work session, 2026-09-15, slide 5
           schools_below_two_2025_26   14.00  BOE work session, 2026-09-15, slide 5
           schools_below_two_2030_31    6.00  BOE work session, 2026-09-15, slide 5
                 utilization_2025_26    0.68  BOE work session, 2026-09-15, slide 5
                 utilization_2030_31    0.75  BOE work session, 2026-09-15, slide 5
            students_per_class_round   25.00  BOE work session, 2026-09-15, slide 5
               two_rounds_enrollment  300.00  BOE work session, 2026-09-15, slide 5
    schools_below_two_2030_no_action   14.00           Resolution 26-27, 2026-09-15
                  decline_since_2017 3675.00           Resolution 26-27, 2026-09-15
                     decline_to_2030 1706.00           Resolution 26-27, 202

### Capacity, and how little of it there is

The work session gives capacity and utilisation for **two schools only** —
Mesa and Bear Creek, on slide 20, because that consolidation was the worked
example. Every other school's capacity is in the deck as a chart, not a
table, so it cannot be read off.

This is the single biggest limit on what follows. Utilisation is the
district's second metric and it cannot be computed here for 29 of the 31
elementary schools.

In [17]:
CAPACITY = [
    ("Bear Creek Elementary School", 492, 275, 312, 0.63),
    ("Mesa Elementary School", 418, 228, 224, 0.54),
]
capacity = pd.DataFrame(CAPACITY, columns=["school_name", "capacity",
                                           "resident_students", "enrolled_students",
                                           "utilization"])
capacity["source"] = "BOE work session, 2026-09-15, slide 20"
capacity["school_code"] = capacity["school_name"].str.upper().map(lookup)
capacity.to_csv(ANALYSIS / "proposal-capacity.csv", index=False)
print(capacity.to_string(index=False))

                 school_name  capacity  resident_students  enrolled_students  utilization                                 source school_code
Bear Creek Elementary School       492                275                312         0.63 BOE work session, 2026-09-15, slide 20        0652
      Mesa Elementary School       418                228                224         0.54 BOE work session, 2026-09-15, slide 20        5838


### Boulder's subcommunities

The City of Boulder's ten subcommunity boundaries, supplied by hand after
the open-data portal proved unreachable. They cover the city only — the
district's other municipalities are not in this layer — so they are used
where they exist and the places layer carries the rest.

In [18]:
subcommunities = gpd.read_file(PROPOSAL / "boulder-subcommunities.geojson").to_crs(4326)
subcommunities = subcommunities[["SUBCOMMUNITY", "geometry"]].rename(
    columns={"SUBCOMMUNITY": "subcommunity"})
subcommunities.to_file(ANALYSIS / "boulder-subcommunities.geojson", driver="GeoJSON")
# The subcommunity goes onto `inside` itself, not a copy: the attendance-area
# step below writes that frame out, and a column added to a copy would vanish.
joined = gpd.sjoin(inside.set_geometry(inside.geometry.representative_point()),
                   subcommunities, how="left", predicate="within")
inside["subcommunity"] = joined["subcommunity"].to_numpy()
by_sub = (inside.groupby("subcommunity", dropna=False)["children"].sum()
          .sort_values(ascending=False))
print(f"{len(subcommunities)} subcommunities; children under 18 by subcommunity:")
print(by_sub.to_string())
inside[["GEOID20", "tract", "block_group", "subcommunity", "population",
        "adults", "children", "ALAND20", "geometry"]].to_file(
    ANALYSIS / "bvsd-blocks.geojson", driver="GeoJSON")

10 subcommunities; children under 18 by subcommunity:
subcommunity
NaN                                  24868
North Boulder                         3072
South Boulder                         2975
Central Boulder                       2453
Palo Park                             2083
Southeast Boulder                     1850
Gunbarrel                             1556
Central Boulder - University Hill     1079
Colorado University                    665
Crossroads                             324
East Boulder                           138


## 8. The district's own attendance areas

The three BVSD attendance-area web maps are ArcGIS web apps with no export
button, but the app's configuration names the web map, the web map names its
layers, and the layers are an ordinary feature service that answers queries.
So the boundaries can be taken properly rather than traced off a screenshot.

The service is `BVSD20FEBEDIT_gdb`, **created and last edited 20 February
2026** — which is the vintage that matters: the resolution says the February
2026 projections were what put 14 elementary schools below the bar by 2030.
These are the boundaries the proposal was drawn against.

The elementary layer carries a **`Capacity` field for all 30 attendance
areas**, which is the figure the work session gives for only two.

In [19]:
BVSD_GIS = ("https://services5.arcgis.com/RovVG757UBJbtq1q/arcgis/rest/services"
            "/BVSD20FEBEDIT_gdb/FeatureServer")
GIS_LAYERS = {
    "elementary-attendance-areas": 7,
    "elementary-attendance-areas-2026": 8,
    "middle-attendance-areas": 0,
    "high-attendance-areas": 1,
    "district-boundary": 3,
    "elementary-locations": 5,
    "middle-locations": 6,
    "high-locations": 2,
}
GEO_BVSD = RAW_GEO / "bvsd"
GEO_BVSD.mkdir(parents=True, exist_ok=True)

def arcgis_geojson(layer: int) -> dict:
    query = (f"{BVSD_GIS}/{layer}/query?where=1%3D1&outFields=*"
             "&outSR=4326&f=geojson")
    request = urllib.request.Request(query, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(request, timeout=300) as response:
        return json.loads(response.read())

gis_manifest = {}
for name, layer in GIS_LAYERS.items():
    path = GEO_BVSD / f"{name}.geojson"
    if not path.exists():
        path.write_text(json.dumps(arcgis_geojson(layer)))
    payload = json.loads(path.read_text())
    gis_manifest[name] = {
        "service": f"{BVSD_GIS}/{layer}",
        "features": len(payload.get("features", [])),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "service_last_edited": "2026-02-20",
    }
    print(f"  {name:36s} {len(payload.get('features', [])):3d} features")
(GEO_BVSD / "manifest.json").write_text(json.dumps(gis_manifest, indent=2, sort_keys=True) + "\n")

  elementary-attendance-areas           30 features
  elementary-attendance-areas-2026      29 features
  middle-attendance-areas               13 features
  high-attendance-areas                  7 features
  district-boundary                      1 features
  elementary-locations                  29 features
  middle-locations                      13 features
  high-locations                         6 features


2362

### Capacity, and what it disagrees with

The layer's `Capacity` is a property of a building and changes slowly. Its
`Enroll` and `StdtPop` are annual figures and disagree with the September
work session for the two schools that can be compared — Mesa's capacity is
485 here and 418 there, its enrollment 371 here and 224 there.

Both are carried. Neither is corrected to the other: they are seven months
and one methodology apart, and which is right is not something this notebook
can settle.

In [20]:
areas = gpd.read_file(GEO_BVSD / "elementary-attendance-areas.geojson").to_crs(4326)
areas = areas.rename(columns={"SchName": "area_name", "SchCode": "area_code",
                              "StdtPop": "resident_students_gis",
                              "Capacity": "capacity_gis", "Enroll": "enrollment_gis",
                              "BVSD_Neigh": "neighborhood_flag"})
areas["area_name"] = areas["area_name"].str.strip()
real = areas[areas["capacity_gis"] > 0].copy()
print(f"{len(areas)} elementary attendance areas; {len(real)} with a capacity")
print(f"total capacity {int(real['capacity_gis'].sum()):,}; "
      f"total enrollment as the layer has it {int(real['enrollment_gis'].sum()):,}; "
      f"utilisation {real['enrollment_gis'].sum() / real['capacity_gis'].sum():.0%}")
stated = figures.set_index("name")["value"]
print(f"the work session gives district utilisation as "
      f"{stated['utilization_2025_26']:.0%} (slide 5)")
print("""
Those two numbers are 18 points apart and they are not the same measure. The
layer's Enroll is whatever it was when the layer was edited; the district's
68% is computed across all its buildings, not only the 27 elementary areas
with a capacity here. Capacity is the field worth taking from this layer -
a building's size does not move much - and it is the only place capacity
exists for more than two schools.
""")
compare = capacity.merge(
    real[["area_name", "capacity_gis", "enrollment_gis", "resident_students_gis"]],
    left_on=capacity["school_name"].str.replace(" Elementary School", "", regex=False),
    right_on="area_name", how="left")
print(compare[["school_name", "capacity", "capacity_gis", "enrolled_students",
               "enrollment_gis", "resident_students", "resident_students_gis"]]
      .to_string(index=False))
areas.to_file(ANALYSIS / "bvsd-elementary-attendance-areas.geojson", driver="GeoJSON")

30 elementary attendance areas; 27 with a capacity
total capacity 12,456; total enrollment as the layer has it 10,717; utilisation 86%
the work session gives district utilisation as 68% (slide 5)

Those two numbers are 18 points apart and they are not the same measure. The
layer's Enroll is whatever it was when the layer was edited; the district's
68% is computed across all its buildings, not only the 27 elementary areas
with a capacity here. Capacity is the field worth taking from this layer -
a building's size does not move much - and it is the only place capacity
exists for more than two schools.

                 school_name  capacity  capacity_gis  enrolled_students  enrollment_gis  resident_students  resident_students_gis
Bear Creek Elementary School       492           475                312             378                275                    169
      Mesa Elementary School       418           485                224             371                228                    258


### Children by attendance area

Every census block is placed in the attendance area that contains it. This
replaces the straight-line nearest-school assignment used earlier: these are
the district's own lines, and where a child goes is decided by the boundary,
not by which school is closest.

In [21]:
points = inside.set_geometry(inside.geometry.representative_point())
placed = gpd.sjoin(points, areas[["area_name", "area_code", "capacity_gis",
                                  "resident_students_gis", "enrollment_gis",
                                  "geometry"]],
                   how="left", predicate="within")
inside["attendance_area"] = placed["area_name"].to_numpy()
matched = inside["attendance_area"].notna()
print(f"{matched.sum():,} of {len(inside):,} blocks fall inside an elementary "
      f"attendance area ({matched.mean():.0%})")
print(f"children in a named area: {int(inside.loc[matched, 'children'].sum()):,} "
      f"of {int(inside['children'].sum()):,}")
by_area = (inside[matched].groupby("attendance_area")
           .agg(blocks=("GEOID20", "size"), children=("children", "sum"))
           .sort_values("children", ascending=False))
print(by_area.head(10).to_string())
inside[["GEOID20", "tract", "block_group", "subcommunity", "attendance_area",
        "population", "adults", "children", "ALAND20", "geometry"]].to_file(
    ANALYSIS / "bvsd-blocks.geojson", driver="GeoJSON")

4,129 of 4,133 blocks fall inside an elementary attendance area (100%)
children in a named area: 41,063 of 41,063
                 blocks  children
attendance_area                  
Crest View          250      3110
Lafayette           177      2350
Ryan                159      2301
Columbine           159      2159
Foothill            186      2145
Louisville          198      1987
Sanchez             165      1979
Creekside           262      1928
Meadowlark          147      1909
Eisenhower          145      1613
